# Etapa 3: Treinamento, Otimização e Avaliação Multicritério do Modelo Preditivo

## 1. Contexto e Formulação Industrial do Problema

Este notebook implementa a arquitetura preditiva oficial do sistema de manutenção baseada em condição da Linha 4-Amarela.

### Duas Visões de Avaliação Operacional:
1. **Visão por Capacidade de Campo (Top-K Prioritário Semanal):**
   * A cada início de ciclo semanal, o modelo ranqueia as 526 portas por Índice de Risco de Quebra.
   * Avalia a **Precisão e Recall** ao dimensionar ordens preventivas para as **Top 5, Top 10, Top 20 ou Top 30 portas mais críticas**.
2. **Visão por Limiar de Certeza (Decisão por Threshold Contínuo):**
   * Disparo automático de chamados quando $P(	ext{Falha}) \ge 	ext{Threshold}$.



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score, f1_score
)
import shap
import joblib
import json
import os
import re
import warnings

warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.dpi'] = 120

print("✅ Bibliotecas carregadas com sucesso!")


## 2. Ingestão da Tabela Mestra e Engenharia de Features Avançada


In [ ]:
# Carregamento da Tabela Mestra Consolidada
df_mestra = pd.read_csv("../../data/tabela_mestra_bruta.csv")
df_mestra['data_ref'] = pd.to_datetime(df_mestra['data_ref'])

# 1. Features de Aceleração e Severidade de Alarmes
df_mestra['aceleracao_alarmes_7d'] = df_mestra['qtd_alarmes_7d'] - (df_mestra['qtd_alarmes_14d'] - df_mestra['qtd_alarmes_7d'])
df_mestra['tempo_medio_por_alarme_7d'] = df_mestra['tempo_parada_acumulado_7d'] / (df_mestra['qtd_alarmes_7d'] + 1)
df_mestra['proporcao_alarmes_graves_7d'] = df_mestra['qtd_anomalias_30s_14d'] / (df_mestra['qtd_alarmes_14d'] + 1)

# 2. Ciclo de Fadiga e Desgaste SAP
df_mestra['faixa_risco_zm01'] = np.where((df_mestra['dias_desde_ultima_zm01'] >= 30) & (df_mestra['dias_desde_ultima_zm01'] <= 90), 1, 0)
df_mestra['fator_risco_ciclo_zm01'] = np.exp(-((df_mestra['dias_desde_ultima_zm01'] - 60) ** 2) / (2 * (30 ** 2)))

# 3. Encodings Estruturais
df_mestra['plataforma_bin'] = df_mestra['plataforma'].map({'P1': 0, 'P2': 1}).fillna(0).astype(int)
df_mestra = pd.get_dummies(df_mestra, columns=['estacao'], prefix='estacao', drop_first=False)

# 4. Divisão Temporal Cega
data_corte = pd.to_datetime('2026-01-01')
df_tr = df_mestra[df_mestra['data_ref'] < data_corte].copy()
df_te = df_mestra[df_mestra['data_ref'] >= data_corte].copy()

cols_features = [
    'idade_ativo_anos', 'porta', 'plataforma_bin', 'faixa_risco_zm01', 'fator_risco_ciclo_zm01',
    'dias_desde_ultima_zm01', 'dias_desde_ultima_zm02', 'dias_desde_ultima_zm03',
    'qtd_alarmes_1d', 'qtd_alarmes_7d', 'qtd_alarmes_14d',
    'aceleracao_alarmes_7d', 'qtd_anomalias_30s_14d',
    'tempo_parada_acumulado_7d', 'max_tempo_normalizacao_7d',
    'tempo_medio_por_alarme_7d', 'proporcao_alarmes_graves_7d'
] + [c for c in df_tr.columns if c.startswith('estacao_')]

scaler = RobustScaler().fit(df_tr[cols_features])
X_tr = scaler.transform(df_tr[cols_features])
X_te = scaler.transform(df_te[cols_features])
y_tr = df_tr['target_falha_7d']
y_te = df_te['target_falha_7d']

print(f"Features de Treinamento ({len(cols_features)}): {cols_features[:6]}...")
print(f"Treino: {X_tr.shape[0]:,} amostras | Teste Cego: {X_te.shape[0]:,} amostras")


## 3. Treinamento do Modelo XGBoost Otimizado


In [ ]:
# Hiperparâmetros consolidados
params = {
    'n_estimators': 300,
    'max_depth': 4,
    'learning_rate': 0.025,
    'min_child_weight': 20,
    'subsample': 0.80,
    'colsample_bytree': 0.80,
    'reg_alpha': 2.0,
    'reg_lambda': 5.0,
    'scale_pos_weight': 9.0,
    'random_state': 42,
    'n_jobs': -1,
    'tree_method': 'hist',
    'eval_metric': 'logloss'
}

model = XGBClassifier(**params)
print("Treinando o Modelo XGBoost...")
model.fit(X_tr, y_tr)

probs_test = model.predict_proba(X_te)[:, 1]
df_te['prob_falha'] = probs_test
df_te['semana'] = df_te['data_ref'].dt.isocalendar().week

# Carregar quebras reais de 2026 para auditoria
df_zm01_raw = pd.read_excel("../../data/Corretiva Emergencial/processados/ZM01_limpa.xlsx")
padrao_sap = re.compile(r'SP04-6-PSD-([A-Z]{3})-(?:PLT|P)(\d+)(?:-PP(\d+))?', re.IGNORECASE)
def extrair_id_sap(loc_str):
    m = padrao_sap.search(str(loc_str))
    if m and m.group(3): return f"{m.group(1).upper()}_P{m.group(2)}_PDM{int(m.group(3)):02d}"
    return None
df_zm01_raw['id_ativo'] = df_zm01_raw['local_de_instalacao'].apply(extrair_id_sap)
df_zm01_raw['data_evento'] = pd.to_datetime(df_zm01_raw['inicio_avaria_datetime'], errors='coerce').dt.normalize()

df_zm01_2026 = df_zm01_raw[
    (df_zm01_raw['data_evento'] >= '2026-01-01') & 
    (df_zm01_raw['data_evento'] <= '2026-02-28') &
    (df_zm01_raw['id_ativo'].notna())
][['id_ativo', 'data_evento']].drop_duplicates().reset_index(drop=True)
df_zm01_2026['semana'] = df_zm01_2026['data_evento'].dt.isocalendar().week

print("✅ Modelo treinado com sucesso!")
print(f"Total de Quebras ZM01 Reais no Período de Teste: {len(df_zm01_2026)} ocorrências.")


## 4. Avaliação Operacional 1: Dimensionamento por Capacidade de Campo (Top-K Semanal)

Esta métrica simula o despacho real das equipes de manutenção ao inspecionar as $K$ portas mais críticas de cada semana ranqueadas pelo modelo.


In [ ]:
df_sem = df_te.groupby(['id_ativo', 'semana']).agg(
    score_risco=('prob_falha', 'max')
).reset_index()

total_quebras = len(df_zm01_2026)

print("=" * 95)
print("🚆 EFICIÊNCIA DE CAMPO: DIMENSIONAMENTO POR TOP-K PORTAS PRIORITÁRIAS POR SEMANA")
print("=" * 95)

resultados_topk = []
for top_k in [5, 10, 15, 20, 30, 40]:
    df_top = df_sem.sort_values(['semana', 'score_risco'], ascending=[True, False]).groupby('semana').head(top_k)
    df_acertos = df_top.merge(df_zm01_2026, on=['id_ativo', 'semana'], how='inner')
    
    chamados_totais = len(df_top)
    acertos_totais = len(df_acertos[['id_ativo', 'semana']].drop_duplicates())
    quebras_evitadas = len(df_acertos['data_evento'].drop_duplicates())
    
    prec_k = acertos_totais / chamados_totais if chamados_totais > 0 else 0
    rec_k = quebras_evitadas / total_quebras if total_quebras > 0 else 0
    f1_k = 2 * prec_k * rec_k / (prec_k + rec_k + 1e-10)
    
    resultados_topk.append({'top_k': top_k, 'chamados': chamados_totais, 'acertos': acertos_totais, 'precisao': prec_k, 'recall': rec_k, 'f1': f1_k})
    print(f"Top {top_k:2d} portas/semana ({chamados_totais:3d} chamados) | Precisão: {prec_k*100:5.2f}% ({acertos_totais:2d} acertos) | Recall: {rec_k*100:5.2f}% ({quebras_evitadas:3d}/{total_quebras} falhas) | F1: {f1_k:.4f}")
print("=" * 95)


## 5. Explicabilidade SHAP e Exportação Final


In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_te)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_te, feature_names=cols_features, show=False, max_display=15)
plt.title('Importância das Features no Modelo Preditivo (SHAP Values)', fontsize=12, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

os.makedirs("../../data/modelos_artefatos", exist_ok=True)
caminho_modelo = "../../data/modelos_artefatos/modelo_xgboost_certeza_absoluta.joblib"
joblib.dump(model, caminho_modelo)
print(f"✅ Modelo preditivo oficial salvo em: {caminho_modelo}")

